# 04 · Eski–Yeni Uydu Köprüsü: 1 Dakikalık Veri Taraması
**Tez:** Güneş Döngüleri 23–25 Boyunca Güneş Patlamaları, Güneş Lekeleri ve Jeomanyetik Etkiler
**Hazırlayan:** Tunahan İnci · **Defter sürümü:** 1.0 (05.11.2026)

**Neden?** 03 defterinde GOES 8–12 grubunu GOES-16 ölçeğine bağlayan tek köprü, GOES-10 ile GOES-14'ün Eylül 2009'daki 5 günlük çakışmasındaki 11 küçük (B sınıfı) patlamaydı. Bu köprüye göre eski uydular %22–25 düşük ölçüyor ve Döngü 25 / Döngü 23 oranı 1,25'ten 0,92'ye düşüyor. Sonuç tamamen bu 11 olaya bağlı.

Flare özet dosyaları yalnızca patlama listesini kapsıyor. Uyduların **1 dakikalık ortalama akı** verileri (`avg1m`) daha uzun dönemleri kapsıyor olabilir. Örneğin GOES-11 veya GOES-12'nin 1 dakikalık verisi 2010–2011'e uzanıyorsa, güneş aktivitesinin yükseldiği bu dönemde GOES-13/14/15 ile yüzlerce C ve M sınıfı ortak patlama çıkar ve köprü sağlamlaşır.

Bu defter **hiçbir şey indirmez**, yalnızca NCEI klasörlerini tarayıp her uydunun 1 dakikalık veri kapsamını ve eski–yeni çakışmalarını raporlar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import re, json, requests, numpy as np, pandas as pd
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor

KOK = Path('/content/drive/MyDrive/gunes_tezi')
HAM, ISLENMIS = KOK/'data'/'raw', KOK/'data'/'processed'
TEMEL = 'https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/science/xrs/'

def linkler(url):
    try:
        r = requests.get(url, timeout=60); r.raise_for_status()
        return [urljoin(url, h) for h in re.findall(r'href="([^"?#]+)"', r.text)
                if not h.startswith(('/', 'http', '..'))]
    except Exception as e:
        print('  erişilemedi:', url, '|', e); return []
print('Hazır')

## 1 · Her uydunun `avg1m` klasörünü tara

In [ ]:
def avg1m_dosyalari(uydu):
    kok = f'{TEMEL}goes{uydu:02d}/'
    alt = [l for l in linkler(kok) if 'avg1m' in l.lower() and l.endswith('/')]
    if not alt:
        print(f'GOES-{uydu:02d}: avg1m klasörü bulunamadı. Klasördekiler:', [l.rsplit("/", 2)[-2] for l in linkler(kok)][:15])
        return uydu, [], []
    dosyalar, kuyruk = [], list(alt)
    while kuyruk:
        u = kuyruk.pop()
        for l in linkler(u):
            if l.endswith('.nc'):
                dosyalar.append(l)
            elif l.endswith('/') and l.startswith(u) and l != u:
                kuyruk.append(l)
    return uydu, alt, dosyalar

with ThreadPoolExecutor(4) as ex:
    sonuc = list(ex.map(avg1m_dosyalari, range(8, 16)))

def tarihler(ad):
    m = re.search(r'_d(\d{8})_', ad)
    if m: return [pd.Timestamp(m.group(1))]
    m = re.search(r'_s(\d{8})_e(\d{8})', ad)
    if m: return list(pd.date_range(m.group(1), m.group(2), freq='D'))
    m = re.search(r'_y(\d{4})_', ad)
    if m: return list(pd.date_range(f'{m.group(1)}-01-01', f'{m.group(1)}-12-31', freq='D'))
    m = re.search(r'_m(\d{6})_', ad)
    if m: return list(pd.date_range(m.group(1) + '01', periods=1, freq='MS').append(
                       pd.date_range(m.group(1) + '01', pd.Timestamp(m.group(1) + '01') + pd.offsets.MonthEnd(0))))
    return []

kapsam = {}
for uydu, alt, dosyalar in sonuc:
    gunler = sorted({d for f in dosyalar for d in tarihler(f.rsplit('/', 1)[1])})
    kapsam[uydu] = pd.DatetimeIndex(gunler)
    ornek = dosyalar[0].rsplit('/', 1)[1] if dosyalar else '-'
    if len(gunler):
        print(f'GOES-{uydu:02d}: {len(dosyalar):5d} dosya | {gunler[0]:%Y-%m-%d} → {gunler[-1]:%Y-%m-%d} | '
              f'{len(gunler):5d} gün | örnek: {ornek}')
    else:
        print(f'GOES-{uydu:02d}: {len(dosyalar)} dosya, tarih çıkarılamadı | örnek: {ornek}')

(HAM/'ncei'/'avg1m_kapsam.json').write_text(json.dumps(
    {u: [str(d.date()) for d in g] for u, g in kapsam.items()}))

## 2 · Eski–yeni çakışmaları
Her eski–yeni uydu çifti için ortak gün sayısı, dönem ve o dönemdeki ortalama güneş lekesi sayısı. Leke sayısı yüksekse o dönemde çok sayıda C/M patlaması vardır ve köprü güçlü olur.

In [ ]:
sn = pd.read_parquet(ISLENMIS/'sn_gunluk.parquet').set_index('tarih').sn
eski, yeni = [8, 9, 10, 11, 12], [13, 14, 15]
satir = []
for a in eski:
    for b in yeni:
        ortak = kapsam.get(a, pd.DatetimeIndex([])).intersection(kapsam.get(b, pd.DatetimeIndex([])))
        if len(ortak):
            satir.append([f'GOES-{a:02d}', f'GOES-{b:02d}', len(ortak), ortak.min().date(), ortak.max().date(),
                          round(sn.reindex(ortak).mean(), 1)])
cakisma = pd.DataFrame(satir, columns=['eski', 'yeni', 'ortak_gun', 'ilk', 'son', 'ort_SN'])
print(cakisma.to_string(index=False) if len(cakisma) else 'Eski ve yeni grup arasında 1 dakikalık veride hiç çakışma yok.')

if len(cakisma):
    print('\nYıllara göre ortak gün sayısı:')
    for r in cakisma.itertuples():
        a, b = int(r.eski[-2:]), int(r.yeni[-2:])
        ortak = kapsam[a].intersection(kapsam[b])
        print(f'  {r.eski} & {r.yeni}:', pd.Series(1, index=ortak).groupby(ortak.year).size().to_dict())